## Introduction to Model Context Protocol (MCP)

![alt text](<Screenshot 2026-10-03 154833.png>)

- Simplifies Connectivity
- Dynamic tools discovery
- MCP Components:
    - Host
    - Client
    - Server

![alt text](<Screenshot 2026-10-03 155750.png>)

MCP primatives:
- Tools
- Resources
- Prompts

### MCP Server

Example code creating MCP Server

In [ ]:
from mcp.server.fastmcp import FastMCP
import requests

# Create an MCP server instance
mcp = FastMCP("Currency Converter")

@mcp.tool()
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """
    Convert an amount from one currency to another using current exchange rates.

    Args:
        amount: The amount to convert
        from_currency: Source currency code (e.g., 'USD', 'EUR', 'GBP')
        to_currency: Target currency code (e.g., 'USD', 'EUR', 'GBP')

    Returns:
        A string with the conversion result and exchange rate
    """
    # API endpoint for Frankfurter
    url = f"https://api.frankfurter.dev/v1/latest?base={from_currency}&symbols={to_currency}"

    try:
        # Make the API request
        response = requests.get(url)
        response.raise_for_status()

        # Parse the response
        data = response.json()

        # Get the exchange rate
        rate = data['rates'].get(to_currency)

        if rate is None:
            return f"Could not find exchange rate for {from_currency} to {to_currency}"

        # Calculate the converted amount
        converted_amount = amount * rate

        return f"{amount} {from_currency} = {converted_amount:.2f} {to_currency} (Rate: {rate})"

    except requests.exceptions.RequestException as e:
        return f"Error converting currency: {str(e)}"

print("Testing Currency Converter:")
result = convert_currency(amount=100, from_currency="USD", to_currency="EUR")
print(result)

### Building MCP Server

In [ ]:
from mcp.server.fastmcp import FastMCP
import requests

mcp = FastMCP("Timezone Converter")

@mcp.tool()
def convert_timezone(date_time, from_timezone, to_timezone):
    """
    Convert a date and time from one timezone to another.

    Args:
        date_time: The date and time to convert (in ISO 8601 format)
        from_timezone: Source timezone (e.g., 'America/New_York')
        to_timezone: Target timezone (e.g., 'Europe/London')
        
    Returns:
        A string with the converted date and time
    """
    url = "https://api.opentimezone.com/convert"
    payload = {
        "dateTime": date_time,
        "fromTimezone": from_timezone,
        "toTimezone": to_timezone
    }
    
    response = requests.post(url, json=payload)
    data = response.json()
    converted_time = data.get("dateTime", "N/A")
    
    return f"Converted time: {converted_time} from {from_timezone} to {to_timezone}"

## MCP Client-Server Communications

![alt text](<Screenshot 2026-10-05 111848.png>)

![alt text](<Screenshot 2026-10-05 111956.png>)

In [ ]:
import mcp from mcp.server.fastmcp import FastMCP
import requests

mcp = FastMCP("")
@mcp.tool()
def get_weather(city: str) -> str:
    """
    Get the current weather for a specified city.

    Args:
        city: The name of the city (e.g., 'London', 'New York')
    """
    
if __name__ == "__main__":
    mcp.run(transport="studio")

### MCP Client: Listing Server Tools

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import sys

async def get_tools_from_mcp():
    # define the server parameters
    params = StdioServerParameters(
        command=sys.executable,
        args=["timezone_server.py"]
    )
    
    # connect to mcp server & open a session
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            # init session
            await session.initialize()
            # ask server what tools it provides
            response = await session.list_tools()
            print("Connected to MCP server!")
            print("Available tools:")
            for tool in response.tools:
                print(f"- {tool.name}: {tool.description}")
                
            return response.tools
        
asyncio.run(get_tools_from_mcp())

### MCP Client: Calling Server Tools

In [ ]:
async def call_mcp_tool(tool_name: str, arguments: dict) -> str:
    params = StdioServerParameters(
        command=sys.executable,
        args=["timezone_server.py"]
        )
        
        # connect to mcp server & open a session
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
                # init session
            await session.initialize()
            
            result = await session.call_tool(tool_name, arguments)
            
            text_content = result.content[0].text
            
            print(f"Conversion Result: {text_content}")
            return str(text_content)
        
# run function
asyncio.run(
    call_mcp_tool(
    tool_name="convert_timezone",
    arguments={
        "date_time": "2024-06-01T12:00:00",
        "from_timezone": "America/New_York",
        "to_timezone": "Europe/London"
    }
    )
)

## Resources in MCP Servers
- Resources (read-only data or data objects retrieved by MCP client, Application-Driven) & prompts
- Integration with APIs & databases

In [ ]:
from mcp.server.fastmcp import FastMCP
import asyncio

mcp = FastMCP("Timezone Converter")

# URI
# can point to local files, APIs, remote servers, etc.
# can dynamically insert parameters like User ID
@mcp.resource("file://locations.txt")
def get_locations():
    try:
        with open("locations.txt", "r") as f:
            content = f.read()
        return content
    except FileNotFoundError:
        return "locations.txt file not found."
    
    
async def list_resources():
    # define the server parameters
    params = StdioServerParameters(
        command=sys.executable,
        args=["timezone_server.py"]
    )
    
    # connect to mcp server & open a session
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            # init session
            await session.initialize()
            response = await session.list_resources()
            
            print("Available resources:")
            for resource in response.resources:
                print(f"- {resource.uri}")
                print(f"  Name: {resource.name}")
                print(f"  Description: {resource.description}")
                
            return response.resources
        
        
async def read_resources(resource_uri: str):
    # define the server parameters
    params = StdioServerParameters(
        command=sys.executable,
        args=["timezone_server.py"]
    )
    
    # connect to mcp server & open a session
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            # init session
            await session.initialize()        
            
            print(f"Reading resource: {resource_uri}")
            resource_content = await session.read_resource(resource_uri)
            
            for content in resource_content.contents:
                print(f"Content: {content.mimeType}")
                print(content.text)
                
            return resource_content

    
if __name__ == "__main__":
    mcp.run(transport="stdio")

### Prompt in MCP Server
- for workflows & behaviors
- reduces prompt-load on users

Example

You are a timezone conversion engine

Your task is to:
- Extract the source datetime from the user's natural language input
- Identify the source timezone 
- Convert the datetime into the target timezone

Rules
- If the date is vague (example, next Friday), resolve it relatively to provided datetime
- If the input cannot be resolved, ask for clarity

### Defining MCP Server Prompt

In [ ]:
@mcp.prompt(title="Timezone Converter", description="Convert date and time between timezones.")
def convert_timezone_prompt(user_input: str) -> str:
    return f"""
You are a timezone conversion engine

Your task is to:
- Extract the source datetime from the user's natural language input
- Identify the source timezone 
- Convert the datetime into the target timezone

Rules
- If the date is vague (example, next Friday), resolve it relatively to provided datetime
- If the input cannot be resolved, ask for clarity

User's timezone conversion request: {user_input}
"""

async def list_prompts():
    # list available prompts from the MCP server
    params = StdioServerParameters(
        command=sys.executable,
        args=["timezone_server.py"]
        )
        
        # connect to mcp server & open a session
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            # init session
            await session.initialize()
            # list available prompts
            prompts = await session.list_prompts()
            print(f"Available prompts: {[p.name for p in prompts.prompts]} ")
            

async def read_prompt(user_input: str, prompt_name: str = "convert_timezone_prompt") -> str:
    # list available prompts from the MCP server
    params = StdioServerParameters(
        command=sys.executable,
        args=["timezone_server.py"]
        )
        
        # connect to mcp server & open a session
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            # init session
            await session.initialize()
            
            prompts = await session.list_prompts()
            # retrieve the prompt by name
            if prompts.prompts:
                prompt = await session.get_prompt(prompt_name, arguments={"timezone_request": user_input})
                print(f"Prompt result: {prompt.messages[0].content.text}")
                
            return prompt.messages[0].content.text
        
        
print(asyncio.run(read_prompt("Convert 2024-06-01 12:00 from New York to London")))

### MCP & LLMs: Tools
- Tool-Calling Workflow
    1. Tools Definitions & User Messages
    2. Tool Call & Arguments
    3. Execute tool
    4. Send Tool Results
    5. Final Response

In [ ]:
from mcp.server.fastmcp import FastMCP
import requests
from antropic import AsyncAnthropic

mcp = FastMCP("Timezone Converter")

@mcp.tool()
def convert_timezone(date_time, from_timezone, to_timezone):
    """
    Convert a date and time from one timezone to another.

    Args:
        date_time: The date and time to convert (in ISO 8601 format)
        from_timezone: Source timezone (e.g., 'America/New_York')
        to_timezone: Target timezone (e.g., 'Europe/London')
        
    Returns:
        A string with the converted date and time
    """
    url = "https://api.opentimezone.com/convert"
    payload = {
        "dateTime": date_time,
        "fromTimezone": from_timezone,
        "toTimezone": to_timezone
    }
    
    response = requests.post(url, json=payload)
    data = response.json()
    converted_time = data.get("dateTime", "N/A")
    
    return f"Converted time: {converted_time} from {from_timezone} to {to_timezone}"

async def get_tools_from_mcp():
    # define the server parameters
    params = StdioServerParameters(
        command=sys.executable,
        args=["timezone_server.py"]
    )
    
    # connect to mcp server & open a session
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            # init session
            await session.initialize()
            # ask server what tools it provides
            response = await session.list_tools()
            print("Connected to MCP server!")
            print("Available tools:")
            for tool in response.tools:
                print(f"- {tool.name}: {tool.description}")
                
            return response.tools
    
async def call_mcp_tool(tool_name: str, arguments: dict) -> str:
    params = StdioServerParameters(
    command=sys.executable,
    args=["timezone_server.py"]
    )
    
        # connect to mcp server & open a session
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
                # init session
            await session.initialize()
            
            result = await session.call_tool(tool_name, arguments)
            
            text_content = result.content[0].text
     
     
            
async def call_antropic_llm(user_query: str) -> str:
    """Call Claude with MCP tools"""
    mcp_tools = await get_tools_from_mcp()
    
    antropic_tools = []
    for tool in mcp_tools:
        antropic_tools.append({
            "name": tool.name,
            "description": tool.description,
            "input_schema": tool.inputSchema # MCP uses JSON Schema format
        })
        
    client = AsyncAnthropic(api_key="YOUR_ANTHROPIC_API_KEY")
    
    response = await client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=300,
        messages=[
            {
                "role":"user", 
                "content":user_query
            }
        ],
        tools=antropic_tools
    )

    if response.stop_reason == "tool_use":
        tool_use = next(b for b in response.content if b.type == "tool_use")
        name = tool_use.tool.name
        args = tool_use.input
        
        print(f"Model decided to call: {name}")
        print(f"With arguments: {args}")
        
        result = await call_mcp_tool(name, args)
        tool_result = {
            "type":"tool_result",
            "tool_use_id": tool_use.id,
            "content":result
            }
        
        followup = await client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=300,
            messages=[
                {
                    "role":"user", 
                    "content":user_query
                },
                {
                    "role":"assistant", 
                    "content": response.content
                },
                {
                    "role":"user", 
                    "content": [tool_result]
                }
            ],
            tools=antropic_tools
        )
        
        final = next((b.text for b in followup.content if b.type == "text"), None)
        if final:
            print(f"Assistant: {final}")
        else:
            print("No final text response from the model.")
    else:
        text = next((b.text for b in response.content if b.type == "text"),"")
        print(f"Assistant: {text}")
        return str(text)

if __name__ == "__main__":
    asyncio.run(call_antropic_llm("Convert 2024-06-01 12:00 from New York to London"))

### MCP & LLMs: Prompt & Resources
- Resources: read-only context
- Prompts: instruction templates to set the behavior & optimize model task
- The Prompt-Resource Workflow

![alt text](<Screenshot 2026-10-06 113855.png>)

In [ ]:
from mcp.server.fastmcp import FastMCP

# tools
@mcp.tool()
def convert_timezone(date_time, from_timezone, to_timezone):
    """
    Convert a date and time from one timezone to another.

    Args:
        date_time: The date and time to convert (in ISO 8601 format)
        from_timezone: Source timezone (e.g., 'America/New_York')
        to_timezone: Target timezone (e.g., 'Europe/London')
        
    Returns:
        A string with the converted date and time
    """
    url = "https://api.opentimezone.com/convert"
    payload = {
        "dateTime": date_time,
        "fromTimezone": from_timezone,
        "toTimezone": to_timezone
    }
    
    response = requests.post(url, json=payload)
    data = response.json()
    converted_time = data.get("dateTime", "N/A")
    
    return f"Converted time: {converted_time} from {from_timezone} to {to_timezone}"

# resources
@mcp.resource("file://locations.txt")
def get_locations():
    try:
        with open("locations.txt", "r") as f:
            content = f.read()
        return content
    except FileNotFoundError:
        return "locations.txt file not found."

# prompt
@mcp.prompt(title="Timezone Converter", description="Convert date and time between timezones.")
def convert_timezone_prompt(user_input: str) -> str:
    return f"""
You are a timezone conversion engine

Your task is to:
- Extract the source datetime from the user's natural language input
- Identify the source timezone 
- Convert the datetime into the target timezone

Rules
- If the date is vague (example, next Friday), resolve it relatively to provided datetime
- If the input cannot be resolved, ask for clarity

User's timezone conversion request: {user_input}
"""

async def list_prompts():
    # list available prompts from the MCP server
    params = StdioServerParameters(
        command=sys.executable,
        args=["timezone_server.py"]
        )
        
        # connect to mcp server & open a session
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            # init session
            await session.initialize()
            # list available prompts
            prompts = await session.list_prompts()
            print(f"Available prompts: {[p.name for p in prompts.prompts]} ")
            
            
# fetch resource & prompt
async def get_context_from_mcp(user_query: str) -> tuple[str,str]:
    """Fetch context from MCP server for a given user query."""
    params = StdioServerParameters(
        command=sys.executable,
        args=["timezone_server.py"]
    )
    
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            await session.initialize()
            
            # Fetch resource content
            resource_result = await session.read_resource("file://locations.txt")
            resource_text = resource_result.content[0].text
            
            # Fetch prompt content
            prompt_result = await session.get_prompt("convert_timezone_prompt", arguments={"timezone_request": user_query})
            prompt_text = prompt_result.messages[0].content.text
            
            return resource_text, prompt_text
        
# build system message & call the LLM
async def call_antropic_llm(user_query: str) -> str:
    """Call Claude with resource & prompt context from MCP server."""
    resource_text, prompt_text = await get_context_from_mcp(user_query)
    
    full_prompt = prompt_text + "Supported locations:\n" + resource_text
    client = AsyncAnthropic(api_key="YOUR_ANTHROPIC_API_KEY")
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=300,
        messages=[
            {
                "role":"user", 
                "content":full_prompt
            }
        ],
        tools=antropic_tools
    )
    
    if response.stop_reason == "end_turn":
        text = next((b.text for b in response.content if b.type == "text"),"")
        print(f"Assistant: {text}")
        return str(text)
    
    if response.stop_reason == "tool_use":
        tool_use = next((b.text for b in response.content if b.type == "tool_use"),"")
        result = await call_mcp_tool(tool_use.tool.name, tool_use.input)
        # ... send the tool_result back to the follow-up
        # request, then print the reply as before
    
        
        
            
if __name__ == "__main__":
    mcp.run(transport="stdio")


## MCP Database Integrations

In [ ]:
from mcp.server.fastmcp import FastMCP
import sqlite3

mcp = FastMCP("Timezone Converter")

# create connection to SQLite database
conn = sqlite3.connect("timezones.db")
conn.row_factory = sqlite3.Row  # Enable named column access

@mcp.resource("db://timezones.db")
def get_locations() -> str:
    """
    Fetch all timezone locations from the SQLite database.

    Returns:
        A string representation of all timezone locations.
    """
    try:
        cursor = conn.execute("SELECT timezone FROM locations")
        rows = cursor.fetchall()
        
        return "\n".join(r["timezone"] for r in rows)
    
    except sqlite3.Error as e:
        return f"Database error: {str(e)}"
    

@mcp.tool()
def lookup_location(prefix: str) -> str:
    """
    Lookup timezone locations that start with the given prefix.

    Args:
        prefix: The prefix string to search for in timezone locations.

    Returns:
        A string representation of matching timezone locations.
    """
    try:
        cursor = conn.execute(
            "SELECT timezone FROM locations WHERE timezone LIKE ?", (f"{prefix}%",))
        rows = cursor.fetchall()
        return "\n".join(r["timezone"] for r in rows)

    
    except sqlite3.Error as e:
        return f"Database error: {str(e)}"
    
if __name__ == "__main__":
    try:
        mcp.run(transport="stdio")
    finally:
        conn.close()
        
    

### MCP API Integration

In [ ]:
import requests
import os

@mcp.tool()
def convert_timezone(date_time: str, from_timezone: str, to_timezone: str) -> str:
    headers = {
        "Content-Type":"application/json"
    }
    api_key = os.environ.get('TIMEZONE_API_KEY')
    
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    
    payload = {
        "dateTime" : date_time,
        "fromTimezone" : from_timezone,
        "toTimezone" : to_timezone
    }
    
    try:
        r = requests.post("https://api.opentimezone.com/convert", json=payload, timeout=10)
        r.raise_for_status()
        data = r.json()
        converted_time = data.get("dateTime", "N/A")
        
        return f"Time in {to_timezone}: {converted_time}"
        
    except requests.exceptions.RequestException as e:
        return f"Error converting timezone: {e}"

### Third-Party MCP Servers

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

params = StdioServerParameters(
    command = "open-library-server",
    args = []
)

async with stdio_client(params) as (reader, writer):
    async with ClientSession(reader, writer) as session:
        await session.initialize()
        
        response = await session.list_tools()
        print("Tools:", [t.name for t in response.tools])_
        
        result = await session.call_tool("get_book_by_title", {"title" : "Dune"})
        
        text = result.content[0].text
        
        print("Tool Call Result:", text)